In [7]:
import keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from keras.layers import Normalization, Dense, Dropout
from tensorflow.keras.layers import Input, Dense, Dropout, Conv2D, MaxPooling2D, GlobalAveragePooling2D, BatchNormalization, Normalization, Activation

In [8]:
from glob import glob

CAPUCHIN_FILES_PATHS = glob(r"data/Parsed_Capuchinbird_Clips/*.wav")
NON_CAPUCHINE_FILE_PATHS = glob(r"data/Parsed_Not_Capuchinbird_Clips/*.wav")

print(f"Number of Capuchin files: {len(CAPUCHIN_FILES_PATHS)}")
print(f"Number of Non-Capuchin files: {len(NON_CAPUCHINE_FILE_PATHS)}")

Number of Capuchin files: 217
Number of Non-Capuchin files: 593


In [9]:
import random

# We are shuffling because the noise files are currently order by their types in the directory
random.shuffle(NON_CAPUCHINE_FILE_PATHS)

NON_CAPUCHINE_FILE_PATHS = NON_CAPUCHINE_FILE_PATHS[:len(CAPUCHIN_FILES_PATHS)]

print(f"Number of Capuchin files: {len(CAPUCHIN_FILES_PATHS)}")
print(f"Number of Non-Capuchin files: {len(NON_CAPUCHINE_FILE_PATHS)}")

Number of Capuchin files: 217
Number of Non-Capuchin files: 217


In [11]:
import yaml

spec_cfg = None

with open(r"../configs/config.yaml", "r") as f:
  config = yaml.safe_load(f)

  try:
      spec_cfg = config["spec_config"]
      shared = config["shared"]
  except KeyError as e:
      raise KeyError(
          f"config.yaml is missing expected key {e}. Check that 'shared.sr' and 'spec_config' are present."
      ) from e
FILTER_ORDER = spec_cfg["filter_order"]
N_FFT = spec_cfg["n_fft"]
HOP_LENGTH = spec_cfg["hop_length"]
N_MELS = spec_cfg["n_mels"]

WINDOW_SIZE = shared["window_size"]
SR = shared["sr"]

print(f"FILTER_ORDER = {FILTER_ORDER}")
print(f"N_FFT = {N_FFT}")
print(f"HOP_LENGTH = {HOP_LENGTH}")
print(f"N_MELS = {N_MELS}")

print(f"WINDOW_SIZE = {WINDOW_SIZE}")
print(f"SR = {SR}")

FILTER_ORDER = 4
N_FFT = 512
HOP_LENGTH = 256
N_MELS = 64
WINDOW_SIZE = 3.0
SR = 24000


# Models

Udhay's model

In [7]:

NUMBER_OF_CHANNELS = 1

def create_and_compile_model():
  base_model = Sequential([
    Conv2D(filters=32, kernel_size= (3,3), padding='same'),
    BatchNormalization(),
    Activation('relu'),

    Conv2D(filters=32, kernel_size= (3,3), padding='same'),
    BatchNormalization(),
    Activation('relu'),

    MaxPooling2D(2,2),

    Conv2D(filters=64, kernel_size=(3,3), padding='same'),
    BatchNormalization(),
    Activation('relu'),

    Conv2D(filters=64, kernel_size=(3,3), padding='same'),
    BatchNormalization(),
    Activation('relu'),

    MaxPooling2D(2,2),

    Conv2D(filters=128, kernel_size=(3,3), padding='same'),
    BatchNormalization(),
    Activation('relu'),
  ])

  INPUT_SHAPE = (N_MELS, int((WINDOW_SIZE * SR) // HOP_LENGTH) - 1, NUMBER_OF_CHANNELS)

  inputs = keras.Input(shape=INPUT_SHAPE)
  normalization_layer = Normalization(axis=None)

  X = normalization_layer(inputs)
  X = base_model(X)
  X = GlobalAveragePooling2D()(X)
  X = Dense(128, activation="relu")(X)
  X = Dropout(0.4)(X)
  X = Dense(128, activation="relu")(X)
  X = Dropout(0.3)(X)
  X = Dense(64, activation="relu")(X)
  X = Dropout(0.2)(X)
  X = Dense(32, activation="relu")(X)

  outputs = Dense(1, activation="sigmoid")(X)

  bird_model = keras.Model(inputs, outputs)

  bird_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
                   loss=keras.losses.BinaryCrossentropy(),
                   metrics= [
                       keras.metrics.BinaryCrossentropy()
                   ])

  return bird_model, normalization_layer

In [8]:
udhay_model = create_and_compile_model()

SVM Attempt

To use an SVM, we need to either use SKLearn which has an SVM or make it ourselves cause TensorFlow Doesn't have SVMs.

In [ ]:
import sys, os, re, random
from pathlib import Path
import numpy as np
import soundfile as sf
from scipy.fft import dct
 
sys.path.insert(0, str(Path.cwd().parent))
from features.mel_spectogram_generator import MelSpectrogramGen
from training.data_prep.audio_normalisation import AudioNorm
from sklearn.metrics import average_precision_score, roc_auc_score
 
SEED = 0
random.seed(SEED); np.random.seed(SEED)
 
mel = MelSpectrogramGen.from_config()
norm = AudioNorm(window_size=WINDOW_SIZE, sr=SR, hop_size=2.0, clip_pad_max_ratio=0.25)
 
files  = list(CAPUCHIN_FILES_PATHS) + list(NON_CAPUCHINE_FILE_PATHS)
labels = np.array([1] * len(CAPUCHIN_FILES_PATHS) + [0] * len(NON_CAPUCHINE_FILE_PATHS))
 
def load_mono(path):
    y, sr = sf.read(path, dtype="float32", always_2d=True)
    return np.ascontiguousarray(y.mean(axis=1)), sr

cache = [load_mono(f) for f in files]

N_MFCC = 20

def clip_features(logmel):
    """(n_mels, n_frames) log-mel dB -> 100-dim feature vector."""
    M = logmel - logmel.mean(axis=1, keepdims=True)
 

    C = dct(M, type=2, axis=0, norm="ortho")[1:1 + N_MFCC]        
    d = np.diff(C, axis=1)                                      
 
    return np.concatenate([
        C.std(axis=1),                    
        np.percentile(C, 10, axis=1),     
        np.percentile(C, 90, axis=1),     
        np.abs(d).mean(axis=1),           
        d.std(axis=1),                   
    ]).astype(np.float32)           
 
def make_feat(i, training=False, augment=None):
    y, sr = cache[i]
    w, _ = norm.random_clipping(y, sr)
    w = np.ascontiguousarray(w.astype(np.float32))
    if training and augment is not None:
        w = np.ascontiguousarray(augment(samples=w, sample_rate=SR).astype(np.float32))
    return clip_features(mel.generate_mel_spectrogram(mel.bandpass_filter(w)))

groups = [re.sub(r"-\d+\.wav$", "", os.path.basename(f)) for f in files]
uniq = sorted(set(groups)); random.Random(SEED).shuffle(uniq)
val_groups = set(uniq[: max(1, len(uniq) // 5)])
tr_idx = [i for i, g in enumerate(groups) if g not in val_groups]
va_idx = [i for i, g in enumerate(groups) if g in val_groups]
print(f"train {len(tr_idx)} clips | val {len(va_idx)} clips | {len(uniq)} recordings, no overlap")

N_AUG = 8
try:
    from audiomentations import (AddGaussianSNR, ClippingDistortion, Compose,
                                 Gain, LowPassFilter, SevenBandParametricEQ)
    augment = Compose([
        SevenBandParametricEQ(min_gain_db=-8.0, max_gain_db=8.0, p=0.8),
        LowPassFilter(min_cutoff_freq=5000, max_cutoff_freq=11000, p=0.5),
        AddGaussianSNR(min_snr_db=3.0, max_snr_db=30.0, p=0.8),
        Gain(min_gain_db=-12.0, max_gain_db=6.0, p=0.9),
        ClippingDistortion(min_percentile_threshold=0, max_percentile_threshold=10, p=0.15),
    ])
except Exception as e:
    print(f"audiomentations unavailable ({type(e).__name__}); training without waveform augmentation")
    augment = None
 
print(f"building features ({N_AUG} augmented copies per training clip)...")
X_tr = np.stack([make_feat(i, True, augment) for _ in range(N_AUG) for i in tr_idx])
y_tr = np.tile(labels[tr_idx], N_AUG)
X_va = np.stack([make_feat(i) for i in va_idx])
y_va = labels[va_idx]
print("feature matrix:", X_tr.shape, "->", X_va.shape)

import keras
from keras import layers, ops
 
@keras.saving.register_keras_serializable()
class RandomFourierFeatures(layers.Layer):
    """Fixed random projection approximating an RBF kernel. NOT trained."""
    def __init__(self, output_dim=1024, scale=None, **kw):
        super().__init__(**kw)
        self.output_dim, self.scale = output_dim, scale
 
    def build(self, input_shape):
        d = input_shape[-1]
        scale = self.scale or np.sqrt(d / 2.0)         
        self.W = self.add_weight(shape=(d, self.output_dim), trainable=False,
            initializer=keras.initializers.RandomNormal(stddev=1.0 / scale), name="W")
        self.b = self.add_weight(shape=(self.output_dim,), trainable=False,
            initializer=keras.initializers.RandomUniform(0.0, 2 * np.pi), name="b")
 
    def call(self, x):
        return np.sqrt(2.0 / self.output_dim) * ops.cos(ops.matmul(x, self.W) + self.b)
 
    def get_config(self):
        return {**super().get_config(), "output_dim": self.output_dim, "scale": self.scale}
 
C_PARAM = 1.0
svm_tf = keras.Sequential([
    layers.Input(shape=(X_tr.shape[1],)),
    layers.Normalization(axis=-1),                      
    RandomFourierFeatures(output_dim=1024),   
    layers.Dense(1, kernel_regularizer=keras.regularizers.l2(1.0 / C_PARAM)),
], name="rbf_svm")
svm_tf.get_layer(index=0).adapt(X_tr)
 
svm_tf.compile(optimizer=keras.optimizers.Adam(1e-3),
               loss=keras.losses.Hinge(),                
               metrics=[keras.metrics.AUC(curve="PR", name="pr_auc")])
 

svm_tf.fit(X_tr, 2 * y_tr - 1, validation_data=(X_va, 2 * y_va - 1),
           epochs=60, batch_size=64, verbose=0)
s2 = svm_tf.predict(X_va, verbose=0).ravel()
print(f"[keras hinge+L2 SVM] PR-AUC {average_precision_score(y_va, s2):.4f} | "
      f"ROC-AUC {roc_auc_score(y_va, s2):.4f}")
print(f"\nbaseline (always-positive) PR-AUC = {y_va.mean():.4f}")

train 323 clips | val 111 clips | 71 recordings, no overlap
audiomentations unavailable (ModuleNotFoundError); training without waveform augmentation
building features (8 augmented copies per training clip)...
feature matrix: (2584, 100) -> (111, 100)

[keras hinge+L2 SVM] PR-AUC 0.9216 | ROC-AUC 0.9340

baseline (always-positive) PR-AUC = 0.3874


Normal CNN Model

In [ ]:
import math
from typing import Sequence, Tuple

import keras
from keras import layers, ops

# Input shape

def input_shape_from_config(
    sr: int,
    window_size: float,
    n_fft: int,
    hop_length: int,
    n_mels: int,
) -> Tuple[int, int, int]:
    """Input shape of one window, matching `MelSpectrogramGen`.

    That generator frames with `sliding_window_view` and does NOT centre-pad, so
    n_frames = 1 + (n_samples - n_fft) // hop_length.
    With sr=24000, window=3.0, n_fft=512, hop=256 this is (64, 280, 1).

    """
    n_samples = int(sr * window_size)
    n_frames = 1 + (n_samples - n_fft) // hop_length
    return (n_mels, n_frames, 1)


@keras.saving.register_keras_serializable(package="stkilda")
class PerExampleSpecNorm(layers.Layer):

    def __init__(self, epsilon: float = 1e-5, **kwargs):
        super().__init__(**kwargs)
        self.epsilon = epsilon

    def call(self, x):
        per_bin_mean = ops.mean(x, axis=2, keepdims=True)          
        centred = x - per_bin_mean                                 
        global_std = ops.std(centred, axis=(1, 2), keepdims=True)   
        return centred / (global_std + self.epsilon)

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        return {**super().get_config(), "epsilon": self.epsilon}


@keras.saving.register_keras_serializable(package="stkilda")
class DeviceGapAugment(layers.Layer):

    def __init__(
        self,
        p: float = 0.5,
        min_cutoff_frac: float = 0.35,
        max_rolloff_db: float = 30.0,
        min_snr_db: float = 3.0,
        max_snr_db: float = 30.0,
        jitter_db: float = 1.0,
        seed: int | None = None,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.p = p
        self.min_cutoff_frac = min_cutoff_frac
        self.max_rolloff_db = max_rolloff_db
        self.min_snr_db = min_snr_db
        self.max_snr_db = max_snr_db
        self.jitter_db = jitter_db
        self.seed = seed
        self.seed_generator = keras.random.SeedGenerator(seed)

    _DB_TO_LN = math.log(10.0) / 10.0

    def call(self, x, training=None):
        if not training:
            return x

        clean = x
        shape = ops.shape(x)
        batch = shape[0]
        n_bins = x.shape[1]
        g = self.seed_generator

        bin_pos = ops.cast(ops.arange(n_bins), x.dtype) / float(max(n_bins - 1, 1))
        bin_pos = ops.reshape(bin_pos, (1, -1, 1, 1))
        cutoff = keras.random.uniform(
            (batch, 1, 1, 1), self.min_cutoff_frac, 1.0, dtype=x.dtype, seed=g
        )
        slope = keras.random.uniform(
            (batch, 1, 1, 1), 0.0, self.max_rolloff_db, dtype=x.dtype, seed=g
        )
        above = ops.maximum(bin_pos - cutoff, 0.0) / ops.maximum(1.0 - cutoff, 1e-6)
        x = x - slope * above

        ref_db = ops.mean(x, axis=(1, 2, 3), keepdims=True)
        snr_db = keras.random.uniform(
            (batch, 1, 1, 1), self.min_snr_db, self.max_snr_db, dtype=x.dtype, seed=g
        )
        noise_db = ref_db - snr_db
        x = ops.logaddexp(x * self._DB_TO_LN, noise_db * self._DB_TO_LN) / self._DB_TO_LN

        if self.jitter_db > 0.0:
            x = x + keras.random.normal(shape, 0.0, self.jitter_db, dtype=x.dtype, seed=g)

        apply_mask = ops.cast(
            keras.random.uniform((batch, 1, 1, 1), 0.0, 1.0, dtype=x.dtype, seed=g) < self.p,
            x.dtype,
        )
        return apply_mask * x + (1.0 - apply_mask) * clean

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        return {
            **super().get_config(),
            "p": self.p,
            "min_cutoff_frac": self.min_cutoff_frac,
            "max_rolloff_db": self.max_rolloff_db,
            "min_snr_db": self.min_snr_db,
            "max_snr_db": self.max_snr_db,
            "jitter_db": self.jitter_db,
            "seed": self.seed,
        }



def _conv_block(
    x,
    filters: int,
    name: str,
    separable: bool = False,
    pool: Tuple[int, int] = (2, 2),
):
    """Conv -> BN -> ReLU, twice, then max-pool. The whole architecture is this.
    """
    Conv = layers.SeparableConv2D if separable else layers.Conv2D
    for i in (1, 2):
        x = Conv(filters, 3, padding="same", use_bias=False, name=f"{name}_conv{i}")(x)
        x = layers.BatchNormalization(name=f"{name}_bn{i}")(x)
        x = layers.Activation("relu", name=f"{name}_relu{i}")(x)
    if pool is not None:
        x = layers.MaxPooling2D(pool, name=f"{name}_pool")(x)
    return x


def build_penguin_cnn(
    input_shape: Tuple[int, int, int],
    widths: Sequence[int] = (16, 32, 64, 64),
    dropout: float = 0.3,
    separable: bool = False,
    device_gap_augment: bool = True,
    augment_kwargs: dict | None = None,
    name: str = "lp_detector",
) -> keras.Model:

    inp = layers.Input(shape=input_shape, name="log_mel")
    x = inp

    if device_gap_augment:
        x = DeviceGapAugment(name="device_gap_augment", **(augment_kwargs or {}))(x)

    x = PerExampleSpecNorm(name="per_example_norm")(x)

    for i, f in enumerate(widths, start=1):
        x = _conv_block(x, f, name=f"block{i}", separable=separable)

    avg = layers.GlobalAveragePooling2D(name="gap")(x)
    mx = layers.GlobalMaxPooling2D(name="gmp")(x)
    x = layers.Concatenate(name="pool_concat")([avg, mx])
    x = layers.Dropout(dropout, name="head_dropout")(x)
    out = layers.Dense(1, activation="sigmoid", dtype="float32", name="lp_prob")(x)

    return keras.Model(inp, out, name=name)


def compile_penguin_cnn(
    model: keras.Model,
    learning_rate: float = 3e-4,
    label_smoothing: float = 0.05,
) -> keras.Model:
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss=keras.losses.BinaryCrossentropy(label_smoothing=label_smoothing),
        metrics=[
            keras.metrics.AUC(curve="PR", name="pr_auc"),
            keras.metrics.AUC(curve="ROC", name="roc_auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )
    return model


def summarise_cost(model: keras.Model, window_seconds: float = 3.0) -> dict:

    macs = 0
    for layer in model.layers:
        try:
            out = tuple(layer.output.shape)
        except AttributeError:
            continue
        if isinstance(layer, layers.SeparableConv2D):
            _, h, w, c_out = out
            c_in = layer.input.shape[-1]
            k = layer.kernel_size[0] * layer.kernel_size[1]
            macs += h * w * c_in * k + h * w * c_in * c_out
        elif isinstance(layer, layers.Conv2D):
            _, h, w, c_out = out
            c_in = layer.input.shape[-1]
            k = layer.kernel_size[0] * layer.kernel_size[1]
            macs += h * w * c_in * c_out * k
        elif isinstance(layer, layers.Dense):
            macs += layer.input.shape[-1] * out[-1]

    params = int(model.count_params())
    info = {
        "params": params,
        "float32_MB": params * 4 / 1e6,
        "int8_MB": params / 1e6,
        "MMACs_per_window": macs / 1e6,
        "windows_per_second_required": 1.0 / window_seconds,
    }
    print(
        f"{model.name}: {params:,} params "
        f"({info['float32_MB']:.2f} MB float32, ~{info['int8_MB']:.2f} MB int8) | "
        f"{info['MMACs_per_window']:.1f} MMACs per {window_seconds:g} s window"
    )
    print(
        "  budget: inference must finish in < "
        f"{window_seconds:g} s or the audio buffer overflows. "
        "Confirm with benchmark_tflite() on the Pi itself."
    )
    return info



INPUT_SHAPE = input_shape_from_config(
    sr=SR, window_size=WINDOW_SIZE, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS
)
print("model input:", INPUT_SHAPE)

model = compile_penguin_cnn(build_penguin_cnn(INPUT_SHAPE))
model.summary()
summarise_cost(model, window_seconds=WINDOW_SIZE)

model input: (64, 280, 1)


Model: "lp_detector"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ log_mel             │ (None, 64, 280,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ device_gap_augment  │ (None, 64, 280,   │          0 │ log_mel[0][0]     │
│ (DeviceGapAugment)  │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ per_example_norm    │ (None, 64, 280,   │          0 │ device_gap_augme… │
│ (PerExampleSpecNor… │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 64, 280,   │        144 │ per_example_norm… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_bn1          │ (None, 64, 280,   │         64 │ block1_conv1[0][… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_relu1        │ (None, 64, 280,   │          0 │ block1_bn1[0][0]  │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 64, 280,   │      2,304 │ block1_relu1[0][… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_bn2          │ (None, 64, 280,   │         64 │ block1_conv2[0][… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_relu2        │ (None, 64, 280,   │          0 │ block1_bn2[0][0]  │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_pool         │ (None, 32, 140,   │          0 │ block1_relu2[0][… │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv1        │ (None, 32, 140,   │      4,608 │ block1_pool[0][0] │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_bn1          │ (None, 32, 140,   │        128 │ block2_conv1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_relu1        │ (None, 32, 140,   │          0 │ block2_bn1[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv2        │ (None, 32, 140,   │      9,216 │ block2_relu1[0][… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_bn2          │ (None, 32, 140,   │        128 │ block2_conv2[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_relu2        │ (None, 32, 140,   │          0 │ block2_bn2[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 16, 70,    │          0 │ block2_relu2[0][

 Total params: 146,833 (573.57 KB)

 Trainable params: 146,129 (570.82 KB)

 Non-trainable params: 704 (2.75 KB)

lp_detector: 146,833 params (0.59 MB float32, ~0.15 MB int8) | 188.4 MMACs per 3 s window
  budget: inference must finish in < 3 s or the audio buffer overflows. Confirm with benchmark_tflite() on the Pi itself.


{'params': 146833,
 'float32_MB': 0.587332,
 'int8_MB': 0.146833,
 'MMACs_per_window': 188.375168,
 'windows_per_second_required': 0.3333333333333333}

In [ ]:
# Training Loop

import os
import re
import random
import sys
import time
from pathlib import Path

import numpy as np
import soundfile as sf
import tensorflow as tf
import keras

sys.path.insert(0, str(Path.cwd().parent))   
from features.mel_spectogram_generator import MelSpectrogramGen
from training.data_prep.audio_normalisation import AudioNorm
from training.data_prep.spec_augmentation import SpecAugment

SEED = 0
random.seed(SEED); np.random.seed(SEED); keras.utils.set_random_seed(SEED)

BATCH = 32
EPOCHS = 150              
TARGET_RECALL = 0.90      
CHECKPOINT = "lp_best.keras"
TFLITE_PATH = "lp_detector_int8.tflite"
RUN_PRUNING = True        
RUN_CV = False            


mel = MelSpectrogramGen.from_config()
spec_aug = SpecAugment.from_config()
norm = AudioNorm.from_config()

FILES = list(CAPUCHIN_FILES_PATHS) + list(NON_CAPUCHINE_FILE_PATHS)
LABELS = np.array([1] * len(CAPUCHIN_FILES_PATHS) + [0] * len(NON_CAPUCHINE_FILE_PATHS))


def load_mono(path):
    """
    Decode to 1-D float32 at the file's native rate.
    """
    y, sr = sf.read(path, dtype="float32", always_2d=True)
    return np.ascontiguousarray(y.mean(axis=1)), sr


print("decoding...")
CACHE = [load_mono(f) for f in FILES]


def make_spec(i):
    """One training example, straight through the pipeline."""
    y, sr = CACHE[i]
    w, _ = norm.random_clipping(y, sr)                     
    w = np.ascontiguousarray(w.astype(np.float32))
    return mel.generate_mel_spectrogram(mel.bandpass_filter(w))[..., None]


GROUPS = [re.sub(r"-\d+\.wav$", "", os.path.basename(f)) for f in FILES]
UNIQ = sorted(set(GROUPS))
random.Random(SEED).shuffle(UNIQ)


def split_indices(val_groups):
    tr = [i for i, g in enumerate(GROUPS) if g not in val_groups]
    va = [i for i, g in enumerate(GROUPS) if g in val_groups]
    return tr, va


def build_datasets(tr_idx, va_idx):
    X_val = np.stack([make_spec(i) for i in va_idx]).astype("float32")
    y_val = LABELS[va_idx].astype("float32")

    def train_gen():
        order = list(tr_idx)
        random.shuffle(order)
        for i in order:
            yield make_spec(i), np.float32(LABELS[i])

    train_ds = (
        tf.data.Dataset.from_generator(
            train_gen,
            output_signature=(
                tf.TensorSpec(INPUT_SHAPE, tf.float32),
                tf.TensorSpec((), tf.float32),
            ),
        )
        .map(spec_aug.augment, num_parallel_calls=tf.data.AUTOTUNE)  
        .batch(BATCH)                                              
        .prefetch(tf.data.AUTOTUNE)                                 
    )
    return train_ds, (X_val, y_val)


VAL_GROUPS = set(UNIQ[: max(1, len(UNIQ) // 5)])
TR_IDX, VA_IDX = split_indices(VAL_GROUPS)
train_ds, (X_val, y_val) = build_datasets(TR_IDX, VA_IDX)
print(
    f"train {len(TR_IDX)} clips ({int(LABELS[TR_IDX].sum())} pos) | "
    f"val {len(VA_IDX)} clips ({int(LABELS[VA_IDX].sum())} pos) | "
    f"{len(UNIQ)} recordings, no overlap"
)





def class_weights_from_labels(labels):
    """
    Inverse-frequency weights, so a rare positive class still moves the loss.
    """
    labels = np.asarray(labels)
    n, n_pos = len(labels), int(labels.sum())
    n_neg = n - n_pos
    if n_pos == 0 or n_neg == 0:
        return {0: 1.0, 1: 1.0}
    return {0: n / (2.0 * n_neg), 1: n / (2.0 * n_pos)}


def default_callbacks(checkpoint_path=CHECKPOINT, monitor="val_pr_auc",
                      patience=20, reduce_patience=8):
    """
    Checkpoint / LR schedule / early stopping, all driven by val PR-AUC.

    """
    return [
        keras.callbacks.ModelCheckpoint(
            str(checkpoint_path), monitor=monitor, mode="max", save_best_only=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor=monitor, mode="max", factor=0.5, patience=reduce_patience, min_lr=1e-6
        ),
        keras.callbacks.EarlyStopping(
            monitor=monitor, mode="max", patience=patience, restore_best_weights=True
        ),
    ]


def train(model, train_ds, val_data, labels_for_weights=None, epochs=EPOCHS,
          checkpoint_path=CHECKPOINT, callbacks=None, verbose=1):
    """Fit and report the best validation PR-AUC."""
    class_weight = (
        class_weights_from_labels(labels_for_weights)
        if labels_for_weights is not None else None
    )
    history = model.fit(
        train_ds,
        validation_data=val_data,
        epochs=epochs,
        class_weight=class_weight,
        callbacks=callbacks if callbacks is not None else default_callbacks(checkpoint_path),
        verbose=verbose,
    )
    if "val_pr_auc" in history.history:
        print(f"best val PR-AUC: {max(history.history['val_pr_auc']):.4f}")
    return model, history


def cross_validate(n_folds=5, epochs=EPOCHS, verbose=0):
    """
    Grouped k-fold over recordings, reporting mean +/- std of val PR-AUC.
    """
    folds = [set(UNIQ[k::n_folds]) for k in range(n_folds)]
    scores = []
    for k, vg in enumerate(folds, start=1):
        keras.backend.clear_session()
        tr_idx, va_idx = split_indices(vg)
        if LABELS[va_idx].sum() == 0 or LABELS[tr_idx].sum() == 0:
            print(f"fold {k}: skipped (a class is missing from this split)")
            continue
        ds, val = build_datasets(tr_idx, va_idx)
        m = compile_penguin_cnn(build_penguin_cnn(INPUT_SHAPE))
        m, hist = train(m, ds, val, labels_for_weights=LABELS[tr_idx], epochs=epochs,
                        checkpoint_path=f"lp_fold{k}.keras", verbose=verbose)
        scores.append(max(hist.history["val_pr_auc"]))
        print(f"fold {k}: val PR-AUC {scores[-1]:.4f}")
    scores = np.array(scores)
    print(f"CV PR-AUC {scores.mean():.4f} +/- {scores.std():.4f}")
    return scores


def pick_threshold(model_or_scores, X=None, y=None, target_recall=TARGET_RECALL):
    """
    Choose the probability cut-off, and report what it costs.
    """
    if isinstance(model_or_scores, keras.Model):
        scores = model_or_scores.predict(X, verbose=0).ravel()
    else:
        scores = np.asarray(model_or_scores).ravel()
    y = np.asarray(y).ravel()

    order = np.argsort(-scores)
    s_sorted, y_sorted = scores[order], y[order]
    tp = np.cumsum(y_sorted)
    fp = np.cumsum(1 - y_sorted)
    recall = tp / max(y.sum(), 1)
    precision = tp / np.maximum(tp + fp, 1)

    ok = np.where(recall >= target_recall)[0]
    if len(ok) == 0:
        idx = len(scores) - 1
        print(f"WARNING: recall {target_recall:.2f} unreachable on this validation set")
    else:
        idx = ok[np.argmax(precision[ok])]  

    out = {
        "threshold": float(s_sorted[idx]),
        "recall": float(recall[idx]),
        "precision": float(precision[idx]),
        "f1": float(2 * precision[idx] * recall[idx] / max(precision[idx] + recall[idx], 1e-9)),
    }
    print(f"threshold {out['threshold']:.4f} -> precision {out['precision']:.3f}, "
          f"recall {out['recall']:.3f}, F1 {out['f1']:.3f}")
    print("  Deployment note: windows overlap (hop 2 s inside a 3 s window), so a real "
          "call lands in consecutive windows. Requiring 2 consecutive detections on the "
          "Pi cuts the false-alarm rate sharply at almost no cost in recall.")
    return out


class MagnitudePruning(keras.callbacks.Callback):

    def __init__(self, final_sparsity=0.5, epochs=30, frequency=10, verbose=True):
        super().__init__()
        self.final_sparsity = final_sparsity
        self.epochs = epochs
        self.frequency = frequency        
        self.verbose = verbose
        self._epoch = 0

    def _prunable(self):
        for layer in self.model.layers:
            if isinstance(layer, (keras.layers.Conv2D, keras.layers.SeparableConv2D,
                                  keras.layers.Dense)):
                for w in layer.trainable_weights:

                    if len(w.shape) >= 2:
                        yield w

    def _current_sparsity(self):
        t = min(max(self._epoch / max(self.epochs - 1, 1), 0.0), 1.0)
        return self.final_sparsity * (1.0 - (1.0 - t) ** 3)

    def _apply(self, sparsity):
        if sparsity <= 0.0:
            return
        for w in self._prunable():
            v = np.asarray(w)
            k = int(round(sparsity * v.size))
            if k <= 0:
                continue
            thresh = np.partition(np.abs(v).ravel(), k - 1)[k - 1]
            w.assign(np.where(np.abs(v) > thresh, v, 0.0).astype(v.dtype))

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch = epoch

    def on_train_batch_end(self, batch, logs=None):
        if batch % self.frequency == 0:
            self._apply(self._current_sparsity())

    def on_epoch_end(self, epoch, logs=None):
        self._apply(self._current_sparsity())

    def on_train_end(self, logs=None):
        self._apply(self.final_sparsity)
        total = zeros = 0
        for w in self._prunable():
            v = np.asarray(w)
            total += v.size
            zeros += int((v == 0).sum())
        if self.verbose:
            print(f"pruning: {zeros:,}/{total:,} kernel weights are zero "
                  f"({zeros / max(total, 1):.1%} sparsity)")


def prune_and_finetune(model, train_ds, val_data, epochs=30, final_sparsity=0.5,
                       learning_rate=1e-4, checkpoint_path="lp_pruned.keras"):
    """
    Fine-tune under the pruning schedule above, keeping the best epoch.
    """
    dense_score = max(history.history["val_pr_auc"]) if "history" in globals() else None

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss=keras.losses.BinaryCrossentropy(label_smoothing=0.05),
        metrics=[keras.metrics.AUC(curve="PR", name="pr_auc"),
                 keras.metrics.AUC(curve="ROC", name="roc_auc"),
                 keras.metrics.Precision(name="precision"),
                 keras.metrics.Recall(name="recall")],
    )
    hist = model.fit(
        train_ds, validation_data=val_data, epochs=epochs,
        class_weight=class_weights_from_labels(LABELS[TR_IDX]),
        callbacks=[
            MagnitudePruning(final_sparsity=final_sparsity, epochs=epochs),
            keras.callbacks.ModelCheckpoint(str(checkpoint_path), monitor="val_pr_auc",
                                            mode="max", save_best_only=True),
        ],
        verbose=1,
    )
    pruned_score = max(hist.history["val_pr_auc"])
    if dense_score is not None:
        print(f"val PR-AUC: dense {dense_score:.4f} -> pruned {pruned_score:.4f}")
        if pruned_score < dense_score - 0.02:
            print("  WARNING: pruning cost more than 2 points of PR-AUC. "
                  "Lower final_sparsity, or skip pruning - the size saving is not worth this.")
    return model

def export_tflite_int8(model, representative_data, out_path=TFLITE_PATH,
                       n_representative=200, full_integer=False):
    """
    Convert to int8 TFLite 
    """
    rep = np.asarray(representative_data, dtype=np.float32)[:n_representative]

    def representative_dataset():
        for sample in rep:
            yield [sample[None, ...].astype(np.float32)]

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    if full_integer:
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    blob = converter.convert()
    Path(out_path).write_bytes(blob)
    print(f"wrote {out_path} ({len(blob) / 1e6:.2f} MB)")
    return out_path


def evaluate_tflite(tflite_path, X, y):
    """
    Score the validation set through the QUANTISED model and print PR-AUC.
    """
    interp = tf.lite.Interpreter(model_path=str(tflite_path))
    interp.allocate_tensors()
    inp, out = interp.get_input_details()[0], interp.get_output_details()[0]

    scores = np.empty(len(X), dtype=np.float32)
    for i, sample in enumerate(X):
        x = sample[None, ...].astype(np.float32)
        if inp["dtype"] != np.float32:                    
            scale, zero = inp["quantization"]
            x = np.round(x / scale + zero).astype(inp["dtype"])
        interp.set_tensor(inp["index"], x)
        interp.invoke()
        v = interp.get_tensor(out["index"])
        if out["dtype"] != np.float32:
            scale, zero = out["quantization"]
            v = (v.astype(np.float32) - zero) * scale
        scores[i] = float(np.ravel(v)[0])

    y = np.asarray(y).ravel()
    order = np.argsort(-scores)
    tp = np.cumsum(y[order]); fp = np.cumsum(1 - y[order])
    recall = tp / max(y.sum(), 1)
    precision = tp / np.maximum(tp + fp, 1)
    pr_auc = float(np.sum(np.diff(np.concatenate([[0.0], recall])) * precision))
    print(f"quantised model PR-AUC: {pr_auc:.4f}")
    return scores



model, history = train(model, train_ds, (X_val, y_val), labels_for_weights=LABELS[TR_IDX])

float_scores = model.predict(X_val, verbose=0).ravel()
op_point = pick_threshold(float_scores, y=y_val)

if RUN_PRUNING:
    model = prune_and_finetune(model, train_ds, (X_val, y_val))


rep_idx = list(TR_IDX)
random.shuffle(rep_idx)
X_rep = np.stack([make_spec(i) for i in rep_idx[:200]]).astype("float32")

export_tflite_int8(model, X_rep)
q_scores = evaluate_tflite(TFLITE_PATH, X_val, y_val)
pick_threshold(q_scores, y=y_val)                  

if RUN_CV:
    cross_validate()

decoding...
train 350 clips (186 pos) | val 84 clips (31 pos) | 74 recordings, no overlap
Epoch 1/150


c:\Users\Douglas\Documents\Monash BACH\BiOM\StKilda\stkilda\STKilda-first.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


     11/Unknown 15s 402ms/step - loss: 1.6851 - pr_auc: 0.5405 - precision: 0.5536 - recall: 0.3333 - roc_auc: 0.4921

c:\Users\Douglas\Documents\Monash BACH\BiOM\StKilda\stkilda\STKilda-first.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


11/11 ━━━━━━━━━━━━━━━━━━━━ 17s 555ms/step - loss: 1.6851 - pr_auc: 0.5405 - precision: 0.5536 - recall: 0.3333 - roc_auc: 0.4921 - val_loss: 0.6831 - val_pr_auc: 0.3922 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5021 - learning_rate: 3.0000e-04
Epoch 2/150
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 482ms/step - loss: 1.1068 - pr_auc: 0.5562 - precision: 0.5429 - recall: 0.5108 - roc_auc: 0.5209 - val_loss: 0.6683 - val_pr_auc: 0.3899 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5262 - learning_rate: 3.0000e-04
Epoch 3/150
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 514ms/step - loss: 0.8760 - pr_auc: 0.6646 - precision: 0.6380 - recall: 0.5591 - roc_auc: 0.6286 - val_loss: 0.6626 - val_pr_auc: 0.3863 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5423 - learning_rate: 3.0000e-04
Epoch 4/150
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 514ms/step - loss: 0.8149 - pr_auc: 0.6706 - precision: 0.6446 - recall: 0.5753 - roc_auc: 0.6723 - val_loss: 0.6620 - v

...

Saved artifact at 'C:\Users\Douglas\AppData\Local\Temp\tmp3azzqxfw'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 280, 1), dtype=tf.float32, name='log_mel')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2014917217168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2014939515600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2014939515408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2014939515792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2014939514832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2014939515024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2014939516560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2014939516368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2014939516752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2014939514640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  201493951

c:\Users\Douglas\Documents\Monash BACH\BiOM\StKilda\stkilda\STKilda-first.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


wrote lp_detector_int8.tflite (0.17 MB)


c:\Users\Douglas\Documents\Monash BACH\BiOM\StKilda\stkilda\STKilda-first.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


latency: mean 3.9 ms | p95 4.6 ms | max 5.4 ms | budget 3000 ms
quantised model PR-AUC: 0.9643
threshold 0.7031 -> precision 1.000, recall 0.903, F1 0.949
  Deployment note: windows overlap (hop 2 s inside a 3 s window), so a real call lands in consecutive windows. Requiring 2 consecutive detections on the Pi cuts the false-alarm rate sharply at almost no cost in recall.


Perch 2.0

In [4]:
from huggingface_hub import snapshot_download
import tensorflow as tf, numpy as np

path = snapshot_download("cgeorgiaw/Perch", local_dir="perch_v2_model")
model = tf.saved_model.load(path)
print(list(model.signatures))

Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 11.58it/s]


['serving_default']


In [ ]:
import os, re, glob, random, numpy as np, soundfile as sf, scipy.signal
import onnxruntime as ort
from huggingface_hub import hf_hub_download
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

PERCH_SR, PERCH_SAMPLES, SEED = 32000, 5 * 32000, 0
CACHE_NPZ = "perch_capuchin_cache.npz"

onnx_path = hf_hub_download("justinchuby/Perch-onnx", "perch_v2.onnx",
                            local_dir="perch_onnx")     
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
print("inputs :", [(i.name, i.shape) for i in sess.get_inputs()])
print("outputs:", [(o.name, o.shape) for o in sess.get_outputs()])


CAPUCHIN_IDX = None
for lf in glob.glob(os.path.join("perch_v2_model", "assets", "*")):
    try:
        rows = [l.strip() for l in open(lf, encoding="utf-8").read().splitlines() if l.strip()]
    except Exception:
        continue
    hits = [i for i, r in enumerate(rows)
            if "perissocephalus" in r.lower() or "capuchinbird" in r.lower()]
    if hits and len(rows) > 14000:          
        CAPUCHIN_IDX = hits[0]
        print(f"capuchinbird: index {CAPUCHIN_IDX} in {os.path.basename(lf)} -> {rows[hits[0]]}")
        break
if CAPUCHIN_IDX is None:
    print("capuchinbird not found in assets - zero-shot skipped.")

def to_perch_window(path):
    y, sr = sf.read(path, dtype="float32", always_2d=True)
    y = y.mean(axis=1)
    if sr != PERCH_SR:
        g = np.gcd(int(sr), PERCH_SR)
        y = scipy.signal.resample_poly(y, PERCH_SR // g, sr // g)
    if len(y) < PERCH_SAMPLES:
        pad = PERCH_SAMPLES - len(y)
        y = np.pad(y, (pad // 2, pad - pad // 2))
    else:
        s = (len(y) - PERCH_SAMPLES) // 2
        y = y[s:s + PERCH_SAMPLES]
    return np.ascontiguousarray(y, dtype=np.float32)

FILES  = list(CAPUCHIN_FILES_PATHS) + list(NON_CAPUCHINE_FILE_PATHS)
LABELS = np.array([1]*len(CAPUCHIN_FILES_PATHS) + [0]*len(NON_CAPUCHINE_FILE_PATHS))

if os.path.exists(CACHE_NPZ):
    z = np.load(CACHE_NPZ); E, L = z["E"], z["L"]
    print("loaded cached embeddings:", E.shape)
else:
    E, L, BATCH = [], [], 8
    for start in range(0, len(FILES), BATCH):
        chunk = np.stack([to_perch_window(f) for f in FILES[start:start + BATCH]])
        emb, lab = sess.run(["embedding", "label"], {"inputs": chunk})
        E.append(emb); L.append(lab)
        print(f"  embedded {min(start + BATCH, len(FILES))}/{len(FILES)}", end="\r")
    E, L = np.concatenate(E), np.concatenate(L)
    np.savez_compressed(CACHE_NPZ, E=E, L=L)
    print("\nembeddings:", E.shape, "| logits:", L.shape)

groups = [re.sub(r"-\d+\.wav$", "", os.path.basename(f)) for f in FILES]
uniq = sorted(set(groups)); random.Random(SEED).shuffle(uniq)
val_groups = set(uniq[: max(1, len(uniq) // 5)])
tr = [i for i, g in enumerate(groups) if g not in val_groups]
va = [i for i, g in enumerate(groups) if g in val_groups]
print(f"train {len(tr)} clips ({LABELS[tr].sum()} pos) | val {len(va)} clips ({LABELS[va].sum()} pos)")
print(f"\nbaseline (always-positive) PR-AUC = {LABELS[va].mean():.4f}")

if CAPUCHIN_IDX is not None:
    s0 = L[va, CAPUCHIN_IDX]
    print(f"[zero-shot logit]  PR-AUC {average_precision_score(LABELS[va], s0):.4f} | "
          f"ROC-AUC {roc_auc_score(LABELS[va], s0):.4f}")

probe = LogisticRegression(max_iter=5000, class_weight="balanced", C=1.0)
probe.fit(E[tr], LABELS[tr])
s1 = probe.predict_proba(E[va])[:, 1]
print(f"[linear probe]     PR-AUC {average_precision_score(LABELS[va], s1):.4f} | "
      f"ROC-AUC {roc_auc_score(LABELS[va], s1):.4f}")

inputs : [('inputs', ['batch', 160000])]
outputs: [('embedding', ['batch', 1536]), ('spatial_embedding', ['batch', 16, 4, 1536]), ('spectrogram', ['batch', 500, 128]), ('label', ['batch', 14795])]
capuchinbird: index 9922 in labels.csv -> Perissocephalus tricolor
  embedded 434/434
embeddings: (434, 1536) | logits: (434, 14795)
train 334 clips (189 pos) | val 100 clips (28 pos)

baseline (always-positive) PR-AUC = 0.2800
[zero-shot logit]  PR-AUC 0.9108 | ROC-AUC 0.9737
[linear probe]     PR-AUC 1.0000 | ROC-AUC 1.0000


BirdNET

Installs BirdNET

In [ ]:
# %pip install "birdnet-analyzer[embeddings,train]"

In [13]:
import os, re, glob, random, numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

def max_capuchin_conf(csv_dir):
    """Per source file: the highest capuchinbird confidence across its 3 s segments."""
    out = {}
    for c in glob.glob(os.path.join(csv_dir, "**", "*.csv"), recursive=True):
        df = pd.read_csv(c)
        col = next((x for x in df.columns if "common" in x.lower() or "species" in x.lower()), None)
        conf = next((x for x in df.columns if "conf" in x.lower()), None)
        stem = os.path.basename(c).split(".BirdNET")[0]
        hit = df[df[col].astype(str).str.contains("Capuchinbird", case=False, na=False)]
        out[stem] = float(hit[conf].max()) if len(hit) else 0.0
    return out

scores = {**max_capuchin_conf("birdnet_out/pos"), **max_capuchin_conf("birdnet_out/neg")}

FILES  = list(CAPUCHIN_FILES_PATHS) + list(NON_CAPUCHINE_FILE_PATHS)
LABELS = np.array([1]*len(CAPUCHIN_FILES_PATHS) + [0]*len(NON_CAPUCHINE_FILE_PATHS))
s = np.array([scores.get(os.path.splitext(os.path.basename(f))[0], 0.0) for f in FILES])

groups = [re.sub(r"-\d+\.wav$", "", os.path.basename(f)) for f in FILES]
uniq = sorted(set(groups)); random.Random(0).shuffle(uniq)
val = set(uniq[:max(1, len(uniq)//5)])
va = [i for i,g in enumerate(groups) if g in val]

print(f"baseline PR-AUC {LABELS[va].mean():.4f}")
print(f"[BirdNET zero-shot] PR-AUC {average_precision_score(LABELS[va], s[va]):.4f} | "
      f"ROC-AUC {roc_auc_score(LABELS[va], s[va]):.4f}")

baseline PR-AUC 0.2800
[BirdNET zero-shot] PR-AUC 0.2800 | ROC-AUC 0.5000


In [ ]:
import os, re, glob, random, numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

BN_OUT = r"C:\Users\Douglas\Documents\Monash BACH\BiOM\StKilda\stkilda\birdnet_out"


csvs = glob.glob(os.path.join(BN_OUT, "**", "*.BirdNET.results.csv"), recursive=True)
print(len(csvs), "result files")
probe = pd.read_csv(csvs[0])
print("columns:", probe.columns.tolist())

SPECIES_COL = next(c for c in probe.columns if "common" in c.lower() or "species" in c.lower())
CONF_COL    = next(c for c in probe.columns if "conf" in c.lower())


scores, all_species = {}, []
for c in csvs:
    df = pd.read_csv(c)
    if df.empty:
        scores[os.path.basename(c).split(".BirdNET")[0]] = 0.0
        continue
    all_species.append(df[SPECIES_COL])
    hit = df[df[SPECIES_COL].astype(str).str.contains("capuchin", case=False, na=False)]
    scores[os.path.basename(c).split(".BirdNET")[0]] = float(hit[CONF_COL].max()) if len(hit) else 0.0

species = pd.concat(all_species) if all_species else pd.Series(dtype=str)
print("\ndistinct species detected:", species.nunique())
print(species.value_counts().head(15))
print("\ncapuchinbird detections:", int(species.astype(str).str.contains("capuchin", case=False).sum()))

FILES  = list(CAPUCHIN_FILES_PATHS) + list(NON_CAPUCHINE_FILE_PATHS)
LABELS = np.array([1]*len(CAPUCHIN_FILES_PATHS) + [0]*len(NON_CAPUCHINE_FILE_PATHS))
s = np.array([scores.get(os.path.splitext(os.path.basename(f))[0], 0.0) for f in FILES])
print(f"\nnon-zero scores: {(s > 0).sum()} of {len(s)}")

groups = [re.sub(r"-\d+\.wav$", "", os.path.basename(f)) for f in FILES]
uniq = sorted(set(groups)); random.Random(0).shuffle(uniq)
val = set(uniq[:max(1, len(uniq)//5)])
va = [i for i, g in enumerate(groups) if g in val]

print(f"baseline PR-AUC {LABELS[va].mean():.4f}")
if (s[va] > 0).any():
    print(f"[BirdNET zero-shot] PR-AUC {average_precision_score(LABELS[va], s[va]):.4f} | "
          f"ROC-AUC {roc_auc_score(LABELS[va], s[va]):.4f}")
else:
    print("[BirdNET zero-shot] no capuchinbird detections - metric undefined, not 0.5")

810 result files
columns: ['Start (s)', 'End (s)', 'Scientific name', 'Common name', 'Confidence', 'File']

distinct species detected: 819
Common name
Capuchinbird             266
Tawny Owl                133
Eurasian Scops-Owl       114
Long-eared Owl            99
Eurasian Eagle-Owl        93
Song Thrush               79
Bushtit                   73
Anna's Hummingbird        62
House Sparrow             57
Italian Sparrow           56
Little Bittern            54
Lyre-tailed Nightjar      53
Little Owl                51
Siren                     51
False Robust Conehead     51
Name: count, dtype: int64

capuchinbird detections: 266

non-zero scores: 217 of 434
baseline PR-AUC 0.2800
[BirdNET zero-shot] PR-AUC 1.0000 | ROC-AUC 1.0000
